# Lost in Translation: Mandarin to Serbian that keeps who's talking

**Track:** AI + Languages · Round 1 is judged blind, so no names or school anywhere in this notebook.

**How to run it:** in Google Colab, click **Runtime → Run all**. It needs no sign-in, no keys and no uploads, and it gives the same numbers every time. A GPU (Runtime → Change runtime type → T4 GPU) makes the translating faster but isn't required.

## 0. Intro

> Our project helps **a Mandarin speaker writing to Serbian colleagues at work** by taking **a Mandarin sentence plus who is speaking, who they're talking to and how formal it is** and giving back **Serbian with the right endings, or a warning when it had to guess**.

**The problem.** In Serbian, the words change with the speaker's gender ("I checked it" is *Проверио сам* from a man, *Проверила сам* from a woman) and with politeness (*ти си* to a friend, *Ви сте* to a boss). Mandarin marks politeness (您 is the polite "you", 你 the casual one) but usually not the speaker's gender. Many translation tools go through English on the way, and English has one "you" and no gendered verbs, so both get lost and the Serbian tends to come out masculine.

**What we compare,** on the same test sentences that we wrote ourselves:
1. **Baseline:** Mandarin → English → Serbian (two Helsinki-NLP models).
2. **Direct:** Meta's NLLB model, Mandarin straight to Serbian, its top answer.
3. **Ours:** NLLB's top 8 answers, plus rules about Serbian endings that pick the one matching the speaker, listener and formality, or say "had to guess" when none does.

## 1. Setup

Loads the libraries we use and fixes the random seed. A "seed" makes anything random (like picking example rows to show) come out the same on every run, so the judges see exactly the numbers in our write-up. Our translations themselves don't use randomness: the models always give the same output for the same input.

*Later, this section will also install the small extras the translation models need (`sentencepiece`, `sacremoses`) and `sacrebleu` for scoring, as in our model-check notebook.*

In [ ]:
# Libraries. Google Colab already has all of these, so nothing needs installing.
import random
import numpy as np                 # fast maths on lists of numbers
import pandas as pd                # tables (like a spreadsheet in Python)
import matplotlib.pyplot as plt    # charts
import sklearn                     # scikit-learn: the machine learning models

# Fix the random seed so every "Run all" gives the same results.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

### Hugging Face token (optional)

We don't need a token: all three models download with no sign-in, and we don't upload anything. This cell is only here in case we ever upload results to Hugging Face. A token would live in Colab's **Secrets** panel (the key icon on the left) under the name `HF_TOKEN`, never in this notebook. Judges won't have it, so everything below works without it.

In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata   # this only exists inside Colab
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass  # not in Colab, or no secret set: that's fine, uploading is just skipped

print("Hugging Face token found." if HF_TOKEN else "No Hugging Face token. That's fine: uploading is skipped.")

### How we'll score the three systems

Every system is scored the same way, on the same **test** rows, so the comparison is fair. The scoring code comes in section 7; here is what each number means:

- **chrF:** a standard translation score (from the `sacrebleu` library). It compares our Serbian to the correct Serbian our team wrote, counting matching letter sequences, from 0 (nothing alike) to 100 (identical). It's good at noticing a wrong ending, because an ending is only a letter or two.
- **Gender correct %:** of the sentences where the speaker's (or listener's) gender shows in Serbian, how many came out with the right gender ending (for example -о vs -ла).
- **Formality correct %:** how many used the right "you": formal *Ви / сте* or casual *ти / си*.
- **Right form in the 8 options %:** how often at least one of NLLB's top 8 answers had the right gender and formality. This is the best our picker could possibly do, so it shows whether a miss is the picker's fault or the model's.
- **Flag accuracy:** when our system says "had to guess", was it really unable to find the right form? And when it doesn't warn, was it right?
- **Worst misses:** 3-5 sentences where it went most wrong, shown side by side with the correct Serbian, and why we think it happened.

## 2. Data

*To do:* Load our test set from the repo's public web address (never a file on someone's laptop):

`https://raw.githubusercontent.com/proelium27/veritas-ai-hackathon/main/data/test_sentences.csv`

Show a few rows and explain each column: the Mandarin sentence, its English meaning, the correct Serbian (Cyrillic), the speaker's gender, the listener's gender, the formality, what the sentence is testing, notes, and whether the row is for **practice** or **test** (the `split` column). Our team wrote and labelled every sentence; they are everyday workplace messages.

## 3. Explore

*To do:* Count the sentences by each label (speaker gender, listener gender, formal or casual) and by practice/test, with a simple bar chart. Check nothing is badly lopsided (for example, almost every speaker a man), because then a system that always says "man" would look good. Read a few rows by eye.

## 4. Split: practice rows and test rows

*To do:* Use the practice/test column exactly as our team set it; we don't reshuffle. About 20 rows are **practice**: we use those to write and check the picker's rules about Serbian endings. The rest are **test**: they're locked away, and **nobody looks at results on the test rows until the rules are final.** Otherwise we could tweak the rules until they fit the test sentences, and the score would look better than the system really is.

## 5. Baseline: through English

*To do:* Translate each sentence Mandarin → English with `Helsinki-NLP/opus-mt-zh-en`, then English → Serbian with `Helsinki-NLP/opus-mt-tc-base-en-sh` (its input starts with `>>srp_Cyrl<<` to ask for Serbian in Cyrillic). This is how many translation apps work. We expect it to lose the speaker's gender and the formal/casual "you", because English doesn't have them. Show the English in the middle too, so you can see where the information disappears.

## 6. Main model: NLLB direct, then NLLB + our picker

*To do, in two steps:*

1. **Direct:** `facebook/nllb-200-distilled-600M` translates Mandarin (`zho_Hans`) straight to Serbian in Cyrillic (`srp_Cyrl`). We take its top answer.
2. **Ours:** ask NLLB for its **top 8** answers instead of one. Our picker reads each answer's Serbian endings (for example -о vs -ла after сам for "I did", Ви/сте vs ти/си for "you") and picks the first one that matches the sentence's known speaker gender, listener gender and formality. If none matches, it keeps NLLB's top answer and marks it **"had to guess"**. The rules were written by our Serbian speaker using only the practice rows. The picker also has to watch for options that have the right ending but the wrong meaning or a non-Serbian word, which we saw in our model check.

## 7. Evaluate

*To do:* On the **test rows only**, score all three systems side by side in one table: chrF, gender correct %, formality correct %, right form in the 8 options %, and flag accuracy (explained in section 1). Then show the 3-5 worst misses: the Mandarin, the correct Serbian, and what each system gave, with our best guess at why it went wrong. We record these numbers, good or bad, in our results log.

## 8. Demo

*To do:* One cell where you type a Mandarin sentence and choose who is speaking (man/woman), who you're talking to (man/woman) and formal or casual. It shows the Serbian from our system, plus a warning when it had to guess. Showing the baseline's answer next to it makes the difference easy to see. This is what we film for the video.

## 9. Limits and credits

**What it can't do:** *to do, from our real results.* Things we already expect: the test set is small (60-80 sentences) and only covers workplace messages; the picker only checks endings it has rules for, so it can miss gender shown in other words; it can only pick from NLLB's 8 options, so if none is right it can't fix the sentence; and the user has to tell it the speaker's and listener's gender, because Mandarin usually doesn't say.

**Credits**
- Data: our own test sentences, written and labelled by our team (`data/test_sentences.csv` in this repo).
- Pre-trained models (from Hugging Face, used as they are):
  - [Helsinki-NLP/opus-mt-zh-en](https://huggingface.co/Helsinki-NLP/opus-mt-zh-en), CC BY 4.0
  - [Helsinki-NLP/opus-mt-tc-base-en-sh](https://huggingface.co/Helsinki-NLP/opus-mt-tc-base-en-sh), CC BY 4.0
  - [facebook/nllb-200-distilled-600M](https://huggingface.co/facebook/nllb-200-distilled-600M), CC BY-NC 4.0 (non-commercial use)
- Libraries: [numpy](https://numpy.org/), [pandas](https://pandas.pydata.org/), [scikit-learn](https://scikit-learn.org/), [matplotlib](https://matplotlib.org/), [transformers](https://github.com/huggingface/transformers), [PyTorch](https://pytorch.org/), [sacrebleu](https://github.com/mjpost/sacrebleu), [sentencepiece](https://github.com/google/sentencepiece), [sacremoses](https://github.com/hplt-project/sacremoses)
- AI tools: *to do (name each tool and what we used it for)*